In [11]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath(".."))

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

df = pd.read_csv("../data/processed_features_v2.csv")

In [12]:
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

X = df.drop(columns=targets + ["Sample Date"])
y = df[targets]

In [13]:
groups = df["Latitude"].round(2).astype(str) + "_" + df["Longitude"].round(2).astype(str)

In [14]:
gkf = GroupKFold(n_splits=5)

scores = []

for train_idx, val_idx in gkf.split(X, y, groups):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    score = r2_score(y_val, preds)

    scores.append(score)

print(scores)
print("Mean CV score:", sum(scores)/len(scores))

[0.22697147625814962, -0.08066072939255871, 0.2591478704403062, 0.13585866370412403, 0.14779109341511107]
Mean CV score: 0.13782167488502645


In [15]:
from datetime import datetime

experiment_id = "rf_groupkfold_v1"

log_line = f"{experiment_id},model_validation,RandomForest,physical_features,n_estimators=200,{round(sum(scores)/len(scores),4)},NA,GroupKFold validation,{datetime.now().date()}"

print(log_line)

rf_groupkfold_v1,model_validation,RandomForest,physical_features,n_estimators=200,0.1378,NA,GroupKFold validation,2026-03-12
